# LAB 01 — Experiments

Notebook cho ba phần đầu của LAB 01: setup dataset, kiểm tra sparse representation và vocabulary inspection.

## 1. Setup and Dataset

Dataset không được commit trong repository. Hãy đặt corpus được cung cấp vào `lab01/data/` hoặc cấu hình `DATA_PATH` trong cell bên dưới. Notebook không tự tìm file bên ngoài project và không giả định format khi dataset chưa được cấu hình.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

# Set this to the provided corpus path after placing it in the project.
# Example: DATA_PATH = Path("data") / "provided-corpus-file"
DATA_PATH = None

DATA_AVAILABLE = False
documents = []
records = []
if DATA_PATH is None:
    print("Dataset not configured. Set DATA_PATH to the provided 30K corpus.")
else:
    DATA_PATH = Path(DATA_PATH)
    if not DATA_PATH.is_file():
        raise FileNotFoundError(f"Configured dataset path does not exist: {DATA_PATH}")
    raise NotImplementedError(
        "Configure the loader after confirming the provided corpus format."
    )


## 2. Experiment 1 — Sparse Representation

Pipeline: raw documents → tokenizer → CountVectorizer → normalized TF → IDF → TF-IDF matrix. The large matrix remains sparse throughout. `CountVectorizer()` uses sklearn's default tokenization and lowercasing behavior.

TODO for the preprocessing ablation: explicitly control lowercasing and tokenization so that Pipeline A/B/C do not apply preprocessing twice or hide it inside the vectorizer.

In [ ]:
if DATA_AVAILABLE:
    try:
        from sklearn.feature_extraction.text import CountVectorizer
        from sklearn.preprocessing import normalize
        from scipy import sparse
        SKLEARN_AVAILABLE = True
    except ImportError as error:
        SKLEARN_AVAILABLE = False
        print(f"scikit-learn is required for Experiment 1: {error}")
else:
    SKLEARN_AVAILABLE = False

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    vectorizer = CountVectorizer()
    count_matrix = vectorizer.fit_transform(documents)
    tf_matrix = normalize(count_matrix, norm="l1", axis=1, copy=True)

    document_frequency = np.asarray(count_matrix.getnnz(axis=0)).ravel()
    number_of_documents = count_matrix.shape[0]
    idf_values = np.log(number_of_documents / document_frequency)
    tfidf_matrix = tf_matrix.multiply(idf_values).tocsr()
    tfidf_matrix.eliminate_zeros()
    feature_names = vectorizer.get_feature_names_out()

    number_of_documents, vocabulary_size = tfidf_matrix.shape
    nnz = tfidf_matrix.nnz
    total_entries = number_of_documents * vocabulary_size
    sparsity = 1 - nnz / total_entries if total_entries else 0.0

    summary = pd.DataFrame({
        "metric": [
            "Number of documents",
            "Vocabulary size",
            "TF-IDF matrix shape",
            "Non-zero entries (nnz)",
            "Sparsity",
        ],
        "value": [
            number_of_documents,
            vocabulary_size,
            tfidf_matrix.shape,
            nnz,
            sparsity,
        ],
    })
    display(summary)
    print(f"Count matrix sparse: {sparse.issparse(count_matrix)}")
    print(f"TF matrix sparse: {sparse.issparse(tf_matrix)}")
    print(f"TF-IDF matrix sparse: {sparse.issparse(tfidf_matrix)}")
    print("CountVectorizer uses sklearn's default tokenization and lowercasing.")
elif not DATA_AVAILABLE:
    print("Experiment 1 skipped because the dataset is not available.")
else:
    print("Experiment 1 skipped because scikit-learn is not available.")


## 3. Vocabulary Inspection

The tables below use the feature ordering returned by the fitted vectorizer. Document frequency means the number of documents containing a term, not the total number of occurrences.

In [ ]:
def get_top_df_terms(document_frequency, feature_names, top_k=20):
    """Return terms ranked by document frequency."""
    table = pd.DataFrame({
        "term": feature_names,
        "document_frequency": document_frequency,
    })
    return table.sort_values("document_frequency", ascending=False).head(top_k).reset_index(drop=True)

def get_top_idf_terms(feature_names, idf, top_k=20):
    """Return terms ranked by the fitted transformer IDF values."""
    table = pd.DataFrame({"term": feature_names, "idf": idf})
    return table.sort_values("idf", ascending=False).head(top_k).reset_index(drop=True)

def get_top_tfidf_terms(tfidf_matrix, feature_names, document_index, top_k=20):
    """Return non-zero TF-IDF terms for one selected document."""
    row = tfidf_matrix.getrow(document_index)
    table = pd.DataFrame({
        "term": feature_names[row.indices],
        "tfidf": row.data,
    })
    return table.sort_values("tfidf", ascending=False).head(top_k).reset_index(drop=True)

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    top_df_terms = get_top_df_terms(document_frequency, feature_names)
    top_idf_terms = get_top_idf_terms(feature_names, idf_values)

    SELECTED_DOC_INDEX = 0
    top_tfidf_terms = get_top_tfidf_terms(
        tfidf_matrix, feature_names, SELECTED_DOC_INDEX
    )

    print("Top 20 terms by document frequency")
    display(top_df_terms)
    print("Top 20 terms by IDF")
    display(top_idf_terms)
    print(f"Top TF-IDF terms in document {SELECTED_DOC_INDEX}")
    print(documents[SELECTED_DOC_INDEX][:500])
    display(top_tfidf_terms)
elif not DATA_AVAILABLE:
    print("Vocabulary inspection skipped because the dataset is not available.")
else:
    print("Vocabulary inspection skipped because scikit-learn is not available.")


### Student analysis

TODO:
- Compare the three term lists.
- Does a frequent corpus term necessarily have high TF-IDF?
- Does a high-IDF term necessarily have high TF-IDF in every document?

## 4. Experiment 2 — Preprocessing Ablation

TODO: implement Pipeline A, Pipeline B and Pipeline C only after the first three sections are reviewed.

## 5. Document Search

TODO: build the query vector and rank documents after preprocessing ablation.

## 6. Evaluation

TODO: add student-provided relevance labels and compute Precision@5, Recall@5 and MRR.

## 7. Error Analysis

TODO: inspect selected good and poor queries after the evaluation set is provided.